In [6]:
# Updated by: Allan Torres
# Date: 04/19/2026
# Course: CS-340-11052-M01

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# changed animal_shelter and AnimalShelter to match current CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelterCRUD  #Updated

###########################
# Data Manipulation / Model
###########################

#Declare username and password variables
username = "aacuser"
password = "YOURPASSWORD" #Developer highly recommend to update this weak password.
                          #It was developer who set it

# Connect to database via CRUD Module
db = AnimalShelterCRUD(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns = ['_id'],inplace = True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare’s logo
image_filename = 'Logo.png' #Grazioso Salvare Logo uploaded to Jupyter Lab
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Place the HTML image tag in the line below into the app.layout code according to your design

#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(
        html.Img(
            src ='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '120px'}
        )
    ),
    # Unique id incluinding dev's name, and date
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center(html.H4('Developed by Allan Torres - 04/19/2026')),
    html.Hr(),
    
    html.Div(
        
        # In code interactive filtering options. Incluiding but not limited to: Radio buttons, drop down, checkboxes, etc.
        [
            html.Label("Rescue Type Filter:", style={'fontWeight': 'bold'}),
            dcc.RadioItems(
                id='filter-type',
                options=[
                    {'label': 'All (Reset)', 'value': 'reset'},
                    {'label': 'Water Rescue', 'value': 'water'},
                    {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                    {'label': 'Disaster/Individual Tracking', 'value': 'disaster'}
                ],
                value='reset',
                labelStyle={'display': 'inline-block', 'marginRight': '15px'}
            )
        ],
        style={'textAlign': 'center'}
    ),

    html.Hr(),
    
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         #Interative Data Table
                         page_size=10,
                         sort_action="native",
                         filter_action="native",
                         row_selectable="single",
                         selected_rows=[0],
                         style_table={'overflowX': 'auto'},
                         style_cell={'textAlign': 'left', 'padding': '5px'},
                         style_header={'backgroundColor': 'lightgrey', 'fontWeight': 'bold'}                         
                        ),
    html.Br(),
    html.Hr(),
    
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

#Helper lists for rescue types
water_breeds = [                #Not really relevant at all, but I 
    "Labrador Retriever Mix",   #prefer to format variables this way: waterBreeds.
    "Chesapeake Bay Retriever",
    "Newfoundland"
]

montain_breeds = [
    "German Shepherd",
    "Border Collie",
    "Australian Shepherd"
]

disaster_breeds = [
    "Doberman Pinscher",
    "Rottweiler",
    "Belgian Malinois"
]


@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    
    # Code to filter interactive data table with MongoDB queries
    if filter_type == 'water':
        records = db.read({
            "age_upon_outcome_in_weeks": {"$lte": 104},
            "breed": {"$in": water_breeds}
        })
        #From here, the conditional chain was generated by Copilot.
        #I am not feeling like writting all this.

    elif filter_type == 'mountain':
        records = db.read({
            "age_upon_outcome_in_weeks": {"$lte": 156},
            "breed": {"$in": montain_breeds}
        })

    elif filter_type == 'disaster':
        records = db.read({
            "age_upon_outcome_in_weeks": {"$lte": 156},
            "breed": {"$in": disaster_breeds}
        })

    else:  # reset
        records = db.read({})

    dff = pd.DataFrame.from_records(records)
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    return dff.to_dict('records')


@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])

def update_graphs(viewData):
    if viewData is None or len(viewData) == 0: #Now we got a nice formatted variable
        return [html.Div("No data available")]
    dff = pd.DataFrame.from_dict(viewData)
    
    fig = px.pie(
        dff,
        names = 'breed',
        title = 'Breed Distribution for Current Filter'
    )
    return [dcc.Graph(figure=fig)]

    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server()


Dash app running on https://fabricbilly-fossilsenior-3000.codio.io/proxy/8050/
